# House Prices Competition - Modeling Notebook

**Objective:** Build and evaluate regression models to predict `SalePrice` (log-transformed)

**Metrics:** RMSE (on log scale) → RMSLE on original scale

**Approach:** Ridge baseline → XGBoost → LightGBM → Ensemble

**Validation:** 5-fold KFold with target encoding inside CV (no leakage)

## 1. Setup & Imports

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Modeling
from sklearn.model_selection import KFold, cross_val_score, cross_val_predict
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, make_scorer
from sklearn.preprocessing import LabelEncoder

# Advanced models
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Hyperparameter tuning
import optuna
from optuna.samplers import TPESampler

# Misc
import joblib
from datetime import datetime

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Load Processed Data

In [8]:
# Load datasets
X_train = pd.read_csv("../data/X_train_processed.csv")
y_train = pd.read_csv("../data/y_train_processed.csv").values.ravel()  # Log-transformed SalePrice
X_test = pd.read_csv("../data/X_test_processed.csv")

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"\ny_train stats - mean: {y_train.mean():.4f}, std: {y_train.std():.4f}")

# %%
# Quick sanity check - no missing values should remain
print("Missing values in X_train:", X_train.isnull().sum().sum())
print("Missing values in X_test:", X_test.isnull().sum().sum())

X_train shape: (1460, 184)
y_train shape: (1460,)
X_test shape: (1459, 184)

y_train stats - mean: 12.0241, std: 0.3993
Missing values in X_train: 0
Missing values in X_test: 0


## 3. Add Outlier Flag Features

In [9]:
def add_outlier_flags(df, suffix=""):
    """Add binary flags for extreme values"""
    df_out = df.copy()
    
    # GrLivArea outlier flag (>4000 sq ft)
    if 'GrLivArea' in df_out.columns:
        df_out[f'IsLargeGrLivArea{suffix}'] = (df_out['GrLivArea'] > 4000).astype(int)
        print(f"  Added IsLargeGrLivArea{suffix}: {(df_out[f'IsLargeGrLivArea{suffix}']==1).sum()} samples")
    
    # TotalBsmtSF outlier flag (>3000 sq ft)
    if 'TotalBsmtSF' in df_out.columns:
        df_out[f'IsLargeTotalBsmtSF{suffix}'] = (df_out['TotalBsmtSF'] > 3000).astype(int)
        print(f"  Added IsLargeTotalBsmtSF{suffix}: {(df_out[f'IsLargeTotalBsmtSF{suffix}']==1).sum()} samples")
    
    # 1stFlrSF outlier flag (>3000 sq ft)
    if '1stFlrSF' in df_out.columns:
        df_out[f'IsLarge1stFlrSF{suffix}'] = (df_out['1stFlrSF'] > 3000).astype(int)
        print(f"  Added IsLarge1stFlrSF{suffix}: {(df_out[f'IsLarge1stFlrSF{suffix}']==1).sum()} samples")
    
    return df_out

In [10]:
print("Adding outlier flags to training data:")
X_train = add_outlier_flags(X_train)

print("\nAdding outlier flags to test data:")
X_test = add_outlier_flags(X_test)


Adding outlier flags to training data:
  Added IsLargeGrLivArea: 0 samples
  Added IsLargeTotalBsmtSF: 0 samples
  Added IsLarge1stFlrSF: 0 samples

Adding outlier flags to test data:
  Added IsLargeGrLivArea: 0 samples
  Added IsLargeTotalBsmtSF: 0 samples
  Added IsLarge1stFlrSF: 0 samples
